In [1]:
import os
import glob
import pandas as pd
import requests
from datetime import datetime

# Словник для зміни індексів NOAA на український алфавіт
noaa_to_ua = {
    1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
    11: 9, 12: 11, 13: 12, 14: 13, 15: 14, 16: 15, 17: 16, 18: 17, 19: 18,
    20: 6, 21: 1, 22: 2, 23: 7, 24: 5, 25: 10
}

# Назви областей для красивого виводу
ua_names = {
    1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька", 5: "Житомирська",
    6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська", 9: "Київська", 10: "Кіровоградська",
    11: "Луганська", 12: "Львівська", 13: "Миколаївська", 14: "Одеська", 15: "Полтавська",
    16: "Рівненська", 17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
    21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська", 25: "Республіка Крим"
}

print("Словники та бібліотеки успішно завантажено!")

Словники та бібліотеки успішно завантажено!


In [1]:
import os
import glob
import pandas as pd
import requests
from datetime import datetime

# Словник для зміни індексів NOAA на український алфавіт
noaa_to_ua = {
    1: 22, 2: 24, 3: 23, 4: 25, 5: 3, 6: 4, 7: 8, 8: 19, 9: 20, 10: 21,
    11: 9, 12: 11, 13: 12, 14: 13, 15: 14, 16: 15, 17: 16, 18: 17, 19: 18,
    20: 6, 21: 1, 22: 2, 23: 7, 24: 5, 25: 10
}

# Назви областей для красивого виводу
ua_names = {
    1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька", 5: "Житомирська",
    6: "Закарпатська", 7: "Запорізька", 8: "Івано-Франківська", 9: "Київська", 10: "Кіровоградська",
    11: "Луганська", 12: "Львівська", 13: "Миколаївська", 14: "Одеська", 15: "Полтавська",
    16: "Рівненська", 17: "Сумська", 18: "Тернопільська", 19: "Харківська", 20: "Херсонська",
    21: "Хмельницька", 22: "Черкаська", 23: "Чернівецька", 24: "Чернігівська", 25: "Республіка Крим"
}

print("Словники та бібліотеки успішно завантажено!")

Словники та бібліотеки успішно завантажено!


In [2]:
os.makedirs("vhi_data", exist_ok=True)

def download_vhi_data():
    # Перевірка: якщо файли вже є, не качаємо знову (захист від колізій)
    if glob.glob("vhi_data/vhi_id_*.csv"):
        print("Файли вже завантажені раніше. Пропускаємо скачування.")
        return

    for province_id in range(1, 26):
        url = f"https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2024&type=Mean"
        response = requests.get(url)
        
        if response.status_code == 200 and "pre" in response.text:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"vhi_data/vhi_id_{province_id}_{timestamp}.csv"
            
            with open(filename, "w", encoding="utf-8") as f:
                f.write(response.text)
            print(f"Завантажено область №{province_id}")

download_vhi_data()

Завантажено область №1
Завантажено область №2
Завантажено область №3
Завантажено область №4
Завантажено область №5
Завантажено область №6
Завантажено область №7
Завантажено область №8
Завантажено область №9
Завантажено область №10
Завантажено область №11
Завантажено область №12
Завантажено область №13
Завантажено область №14
Завантажено область №15
Завантажено область №16
Завантажено область №17
Завантажено область №18
Завантажено область №19
Завантажено область №20
Завантажено область №21
Завантажено область №22
Завантажено область №23
Завантажено область №24
Завантажено область №25


In [5]:
def load_and_clean_data(folder="vhi_data"):
    all_dfs = []
    files = glob.glob(f"{folder}/vhi_id_*.csv")
    
    for file in files:
        orig_id = int(os.path.basename(file).split("_")[2])
        
        # Читаємо файл повністю як текст, щоб акуратно відфільтрувати рядки з тегами та заголовками
        with open(file, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            
        clean_lines = []
        for line in lines:
            # Пропускаємо HTML-теги та мета-інформацію сервера
            if '<pre>' in line or '</pre>' in line or 'html' in line or 'Country' in line:
                continue
            if line.strip():  # додаємо лише непусті рядки
                clean_lines.append(line)
        
        # Створюємо список списків (розбиваємо кожен рядок по комі)
        data = [l.strip().split(',') for l in clean_lines]
        
        # Перетворюємо у DataFrame
        df_temp = pd.DataFrame(data)
        
        # Якщо стовпців виявилося більше ніж 7 через кому в кінці рядка, беремо лише перші 7
        if df_temp.shape[1] >= 7:
            df_temp = df_temp.iloc[:, :7]
            df_temp.columns = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']
        else:
            continue # якщо файл пошкоджений, пропускаємо його
            
        # Очищення від порожніх значень
        df_temp = df_temp.dropna()
        
        # Перетворюємо типи в числові
        for col in df_temp.columns:
            df_temp[col] = pd.to_numeric(df_temp[col], errors='coerce')
        df_temp = df_temp.dropna()
        
        # Додаємо українські індекси та назви
        ua_id = noaa_to_ua.get(orig_id, orig_id)
        df_temp['Area_ID'] = ua_id
        df_temp['Area_Name'] = ua_names.get(ua_id, "Невідомо")
        
        all_dfs.append(df_temp)
        
    main_df = pd.concat(all_dfs, ignore_index=True)
    main_df = main_df[main_df['VHI'] >= 0]
    return main_df

# Перезапускаємо створення головного датафрейму
df = load_and_clean_data()
print("Дані успішно очищено та об'єднано!")
df.head()

Дані успішно очищено та об'єднано!


,Year,Week,SMN,SMT,VCI,TCI,VHI,Area_ID,Area_Name
0,1982.0,2.0,0.063,261.53,55.89,38.20,47.04,21,Хмельницька
1,1982.0,3.0,0.063,263.45,57.30,32.69,44.99,21,Хмельницька
2,1982.0,4.0,0.061,265.10,53.96,28.62,41.29,21,Хмельницька
3,1982.0,5.0,0.058,266.42,46.87,28.57,37.72,21,Хмельницька
4,1982.0,6.0,0.056,267.47,39.55,30.27,34.91,21,Хмельницька


In [6]:
def get_vhi_range(df, provinces, start_year, end_year):
    result = df[(df['Area_ID'].isin(provinces)) & (df['Year'] >= start_year) & (df['Year'] <= end_year)]
    return result[['Year', 'Week', 'Area_ID', 'Area_Name', 'VHI']]

print("Вибірка для Вінницької (1) та Волинської (2) областей за 2021-2022 роки:")
display(get_vhi_range(df, provinces=[1, 2], start_year=2021, end_year=2022).head(10))

Вибірка для Вінницької (1) та Волинської (2) областей за 2021-2022 роки:


,Year,Week,Area_ID,Area_Name,VHI
28847,2021.0,1.0,1,Вінницька,44.63
28848,2021.0,2.0,1,Вінницька,47.66
28849,2021.0,3.0,1,Вінницька,50.60
28850,2021.0,4.0,1,Вінницька,55.30
28851,2021.0,5.0,1,Вінницька,58.79
28852,2021.0,6.0,1,Вінницька,57.59
28853,2021.0,7.0,1,Вінницька,54.25
28854,2021.0,8.0,1,Вінницька,50.09
28855,2021.0,9.0,1,Вінницька,47.19
28856,2021.0,10.0,1,Вінницька,45.81
